# 02 - Qualidade e pré-processamento dos dados

## 1. Objetivo do notebook

Este notebook tem como objetivo avaliar a qualidade da base horária consolidada da estação A701, identificando valores ausentes, códigos especiais, duplicidades, interrupções temporais e diferenças de completude entre variáveis e períodos.

As decisões de tratamento serão definidas somente após a análise dessas características, preservando a distinção entre ausência de chuva e ausência de informação.


## 2. Bibliotecas e carregamento da base Silver

Nesta etapa, são importadas as bibliotecas utilizadas no pré-processamento e carregada a base horária consolidada produzida no notebook anterior, armazenada na camada Silver.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


In [2]:
pasta_silver = Path("../data/silver")

arquivo_silver = pasta_silver / "a701_2010_2025_horario_consolidado.parquet"

print(arquivo_silver)

..\data\silver\a701_2010_2025_horario_consolidado.parquet


In [3]:
df = pd.read_parquet(arquivo_silver)

print(f"Linhas: {df.shape[0]:,}")
print(f"Colunas: {df.shape[1]}")

Linhas: 140,256
Colunas: 20


## 3. Validação estrutural inicial

Nesta etapa, é verificada a estrutura da base consolidada, incluindo tipos das variáveis, intervalo temporal e quantidade de registros.


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 140256 entries, 0 to 140255
Data columns (total 20 columns):
 #   Column                                                 Non-Null Count   Dtype         
---  ------                                                 --------------   -----         
 0   DATA (YYYY-MM-DD)                                      140256 non-null  datetime64[us]
 1   HORA (UTC)                                             140256 non-null  str           
 2   PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                       139540 non-null  float64       
 3   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  139976 non-null  float64       
 4   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)        139972 non-null  float64       
 5   PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)       139972 non-null  float64       
 6   RADIACAO GLOBAL (KJ/m²)                                111257 non-null  float64       
 7   TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)           139976 non-n

In [5]:
print("Data mínima:", df["DATA (YYYY-MM-DD)"].min())
print("Data máxima:", df["DATA (YYYY-MM-DD)"].max())
print("Quantidade de registros:", len(df))
print("Quantidade de colunas:", df.shape[1])

Data mínima: 2010-01-01 00:00:00
Data máxima: 2025-12-31 00:00:00
Quantidade de registros: 140256
Quantidade de colunas: 20


## 4. Análise de valores ausentes e códigos especiais

Nesta etapa, são quantificados os valores ausentes e os códigos especiais presentes nas variáveis meteorológicas. O objetivo é distinguir ausência de informação de valores válidos, especialmente no caso da precipitação, em que o valor zero representa ausência de chuva e não deve ser tratado como dado faltante.


In [6]:
resumo_nulos = pd.DataFrame({
    "nulos": df.isna().sum(),
    "percentual_nulos": (df.isna().mean() * 100).round(2)
})

resumo_nulos = resumo_nulos.sort_values(
    "percentual_nulos",
    ascending=False
)

resumo_nulos

,nulos,percentual_nulos
RADIACAO GLOBAL (KJ/m²),28999,20.68
"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",716,0.51
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C),288,0.21
"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",280,0.20
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C),286,0.20
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%),284,0.20
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB),284,0.20
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB),284,0.20
TEMPERATURA DO PONTO DE ORVALHO (°C),283,0.20
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C),284,0.20


In [7]:
colunas_numericas = df.select_dtypes(include="number").columns

contagem_9999 = (df[colunas_numericas] == -9999).sum()

contagem_9999 = contagem_9999[
    contagem_9999 > 0
].sort_values(ascending=False)

contagem_9999

RADIACAO GLOBAL (KJ/m²)                                  36957
TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)               613
TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)                 517
VENTO, RAJADA MAXIMA (m/s)                                 467
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                   466
TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)           466
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                   466
TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)           466
TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)                 466
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)           464
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)            464
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                           463
TEMPERATURA DO PONTO DE ORVALHO (°C)                       461
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                       461
UMIDADE RELATIVA DO AR, HORARIA (%)                        461
VENTO, VELOCIDADE HORARIA (m/s)                        

### Interpretação da qualidade inicial

A análise mostrou que a base possui dois tipos de ausência de informação: valores nulos (`NaN`) e registros codificados como `-9999`. Como esses dois casos representam indisponibilidade de informação, ambos foram considerados na avaliação de completude.

A precipitação apresentou 1.179 registros horários indisponíveis ao considerar conjuntamente os valores nulos e os códigos `-9999`. As variáveis auxiliares apresentaram baixa proporção de ausência, com exceção da radiação global, que mostrou um padrão de indisponibilidade fortemente relacionado ao horário do dia.

In [8]:
df["ano"] = df["DATA (YYYY-MM-DD)"].dt.year

In [9]:
variaveis_principais = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
    "RADIACAO GLOBAL (KJ/m²)"
]

In [10]:
qualidade_por_ano = []

for ano, grupo in df.groupby("ano"):
    for coluna in variaveis_principais:
        nulos = grupo[coluna].isna().sum()
        cod_9999 = (grupo[coluna] == -9999).sum()

        total_indisponivel = nulos + cod_9999
        percentual = total_indisponivel / len(grupo) * 100

        qualidade_por_ano.append({
            "ano": ano,
            "variavel": coluna,
            "nulos": nulos,
            "codigo_-9999": cod_9999,
            "total_indisponivel": total_indisponivel,
            "percentual_indisponivel": round(percentual, 2)
        })

qualidade_por_ano = pd.DataFrame(qualidade_por_ano)

qualidade_por_ano.head()

,ano,variavel,nulos,codigo_-9999,total_indisponivel,percentual_indisponivel
0,2010,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",0,10,10,0.11
1,2010,"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",0,162,162,1.85
2,2010,"UMIDADE RELATIVA DO AR, HORARIA (%)",0,10,10,0.11
3,2010,"PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORAR...",0,10,10,0.11
4,2010,"VENTO, VELOCIDADE HORARIA (m/s)",0,10,10,0.11


In [11]:
tabela_qualidade_anual = qualidade_por_ano.pivot(
    index="ano",
    columns="variavel",
    values="percentual_indisponivel"
)

tabela_qualidade_anual

variavel,"PRECIPITAÇÃO TOTAL, HORÁRIO (mm)","PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",RADIACAO GLOBAL (KJ/m²),"TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)","UMIDADE RELATIVA DO AR, HORARIA (%)","VENTO, VELOCIDADE HORARIA (m/s)"
ano,,,,,,
2010,0.11,0.11,46.19,1.85,0.11,0.11
2011,0.26,0.26,46.56,0.26,0.26,0.26
2012,0.43,0.43,46.35,0.43,0.43,0.43
2013,0.53,0.53,46.15,0.53,0.53,0.53
2014,2.33,2.33,47.72,2.33,2.33,2.33
2015,0.00,0.00,46.27,0.00,0.00,0.00
2016,0.75,0.75,47.68,0.76,0.76,0.75
2017,0.83,0.83,47.64,0.83,0.83,0.84
2018,0.03,0.00,47.08,0.00,0.00,0.00


## 5. Padronização temporal e distribuição das falhas

A análise anual e mensal mostrou que as falhas de precipitação não estão distribuídas uniformemente ao longo da série. Alguns períodos concentram maior indisponibilidade, com destaque para 2014 e, principalmente, 2023.

Antes de investigar os blocos consecutivos de falha, é necessário garantir uma representação temporal única para todos os anos da série.


In [12]:
df["datetime"] = pd.to_datetime(
    df["DATA (YYYY-MM-DD)"].astype(str)
    + " "
    + df["HORA (UTC)"].astype(str),
    errors="coerce"
)

print("Datetimes inválidos:", df["datetime"].isna().sum())

Datetimes inválidos: 61368


### Diagnóstico do formato de hora

A tentativa inicial de combinar diretamente os campos de data e hora resultou em 61.368 timestamps inválidos. A inspeção mostrou que o formato do campo de hora mudou a partir de 2019, passando de `HH:MM` para `HHMM UTC`.

Por esse motivo, o campo de hora foi padronizado antes da criação definitiva do timestamp. Esse diagnóstico foi mantido no notebook para documentar a inconsistência encontrada na fonte e a decisão adotada para corrigi-la.


In [13]:
for ano, grupo in df.groupby("ano"):
    primeira_hora = grupo["HORA (UTC)"].iloc[0]

    print(
        f"{ano} -> {primeira_hora}"
    )

2010 -> 00:00
2011 -> 00:00
2012 -> 00:00
2013 -> 00:00
2014 -> 00:00
2015 -> 00:00
2016 -> 00:00
2017 -> 00:00
2018 -> 00:00
2019 -> 0000 UTC
2020 -> 0000 UTC
2021 -> 0000 UTC
2022 -> 0000 UTC
2023 -> 0000 UTC
2024 -> 0000 UTC
2025 -> 0000 UTC


In [14]:
def padronizar_hora(valor):
    valor = str(valor).strip()

    if "UTC" in valor:
        valor = valor.replace(" UTC", "")
        return f"{valor[:2]}:{valor[2:]}"

    return valor

In [15]:
df["hora_padronizada"] = df["HORA (UTC)"].apply(padronizar_hora)

In [16]:
for ano, grupo in df.groupby("ano"):
    print(
        ano,
        "-> original:",
        grupo["HORA (UTC)"].iloc[0],
        "| padronizada:",
        grupo["hora_padronizada"].iloc[0]
    )

2010 -> original: 00:00 | padronizada: 00:00
2011 -> original: 00:00 | padronizada: 00:00
2012 -> original: 00:00 | padronizada: 00:00
2013 -> original: 00:00 | padronizada: 00:00
2014 -> original: 00:00 | padronizada: 00:00
2015 -> original: 00:00 | padronizada: 00:00
2016 -> original: 00:00 | padronizada: 00:00
2017 -> original: 00:00 | padronizada: 00:00
2018 -> original: 00:00 | padronizada: 00:00
2019 -> original: 0000 UTC | padronizada: 00:00
2020 -> original: 0000 UTC | padronizada: 00:00
2021 -> original: 0000 UTC | padronizada: 00:00
2022 -> original: 0000 UTC | padronizada: 00:00
2023 -> original: 0000 UTC | padronizada: 00:00
2024 -> original: 0000 UTC | padronizada: 00:00
2025 -> original: 0000 UTC | padronizada: 00:00


In [17]:
df["datetime"] = pd.to_datetime(
    df["DATA (YYYY-MM-DD)"].dt.strftime("%Y-%m-%d")
    + " "
    + df["hora_padronizada"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

print("Datetimes inválidos:", df["datetime"].isna().sum())
print("Datetime mínimo:", df["datetime"].min())
print("Datetime máximo:", df["datetime"].max())

Datetimes inválidos: 0
Datetime mínimo: 2010-01-01 00:00:00
Datetime máximo: 2025-12-31 23:00:00


In [18]:
col_precipitacao = "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)"

df["precipitacao_indisponivel"] = (
    df[col_precipitacao].isna()
    | (df[col_precipitacao] == -9999)
)

In [19]:
falhas_precipitacao_ano = (
    df.groupby("ano")["precipitacao_indisponivel"]
      .agg(["sum", "mean"])
)

falhas_precipitacao_ano["percentual"] = (
    falhas_precipitacao_ano["mean"] * 100
).round(2)

falhas_precipitacao_ano[["sum", "percentual"]]

,sum,percentual
ano,,
2010,10,0.11
2011,23,0.26
2012,38,0.43
2013,46,0.53
2014,204,2.33
2015,0,0.00
2016,66,0.75
2017,73,0.83
2018,3,0.03


In [20]:
falhas_precipitacao_mes = (
    df[df["precipitacao_indisponivel"]]
    .groupby([
        df["datetime"].dt.year.rename("ano"),
        df["datetime"].dt.month.rename("mes")
    ])
    .size()
    .unstack(fill_value=0)
)

falhas_precipitacao_mes

mes,1,2,3,4,5,6,7,8,9,10,11,12
ano,,,,,,,,,,,,
2010,2,0,0,0,0,0,2,3,2,0,1,0
2011,0,0,10,0,3,4,3,2,0,0,1,0
2012,0,0,25,0,1,1,5,3,1,2,0,0
2013,0,0,18,1,0,2,12,7,3,1,2,0
2014,0,1,0,0,0,0,203,0,0,0,0,0
2016,0,66,0,0,0,0,0,0,0,0,0,0
2017,0,0,0,0,73,0,0,0,0,0,0,0
2018,0,0,0,0,1,0,0,2,0,0,0,0
2020,0,0,0,0,0,0,0,1,0,1,0,0


In [21]:
falhas = df.loc[
    df["precipitacao_indisponivel"],
    ["datetime"]
].copy()

falhas["diferenca_horas"] = falhas["datetime"].diff().dt.total_seconds().div(3600)

falhas["novo_bloco"] = falhas["diferenca_horas"].ne(1)

falhas["bloco"] = falhas["novo_bloco"].cumsum()

blocos_falha = (
    falhas.groupby("bloco")
    .agg(
        inicio=("datetime", "min"),
        fim=("datetime", "max"),
        horas_faltantes=("datetime", "size")
    )
    .sort_values("horas_faltantes", ascending=False)
)

blocos_falha.head(20)

,inicio,fim,horas_faltantes
bloco,,,
109,2023-09-03 13:00:00,2023-09-12 15:00:00,219
108,2023-08-24 21:00:00,2023-09-02 20:00:00,216
67,2014-07-20 06:00:00,2014-07-28 16:00:00,203
68,2016-02-08 21:00:00,2016-02-11 14:00:00,66
119,2023-11-26 07:00:00,2023-11-28 12:00:00,54
70,2017-05-20 22:00:00,2017-05-22 16:00:00,43
69,2017-05-19 07:00:00,2017-05-20 12:00:00,30
118,2023-11-21 10:00:00,2023-11-22 13:00:00,28
25,2012-03-07 13:00:00,2012-03-08 12:00:00,24


### Interpretação das falhas de precipitação

Foram identificados blocos consecutivos de ausência com duração elevada, incluindo períodos superiores a 200 horas em 2014 e 2023.

Esse padrão indica que parte das falhas corresponde a interrupções prolongadas de medição, e não apenas a ausências pontuais. Por esse motivo, optou-se por não realizar imputação da precipitação horária nesses períodos, evitando a criação artificial de valores para a variável-alvo.

## 6. Completude e tratamento da precipitação

Como a variável-alvo do estudo será o acumulado diário de precipitação, a qualidade dos registros horários é avaliada antes da agregação. Para cada dia, verifica-se a quantidade de observações horárias válidas, evitando que dias incompletos sejam interpretados como dias secos ou tenham seu acumulado subestimado.


In [22]:
qualidade_diaria_precipitacao = (
    df.assign(
        precipitacao_valida=(
            df[col_precipitacao].notna()
            & (df[col_precipitacao] != -9999)
        )
    )
    .groupby("DATA (YYYY-MM-DD)")
    .agg(
        horas_registradas=("datetime", "size"),
        horas_validas_precipitacao=("precipitacao_valida", "sum")
    )
)

qualidade_diaria_precipitacao["horas_indisponiveis"] = (
    qualidade_diaria_precipitacao["horas_registradas"]
    - qualidade_diaria_precipitacao["horas_validas_precipitacao"]
)

qualidade_diaria_precipitacao.head()

,horas_registradas,horas_validas_precipitacao,horas_indisponiveis
DATA (YYYY-MM-DD),,,
2010-01-01,24,24,0
2010-01-02,24,24,0
2010-01-03,24,24,0
2010-01-04,24,24,0
2010-01-05,24,24,0


In [23]:
qualidade_diaria_precipitacao["horas_indisponiveis"].value_counts().sort_index()

horas_indisponiveis
0     5655
1      103
2       17
3       11
4        3
5        2
6        1
7        2
9        3
10       1
11       3
12       1
13       2
14       2
15       4
16       1
17       4
18       1
21       1
24      27
Name: count, dtype: int64

In [24]:
total_dias = len(qualidade_diaria_precipitacao)

dias_completos = (
    qualidade_diaria_precipitacao["horas_indisponiveis"] == 0
).sum()

dias_incompletos = (
    qualidade_diaria_precipitacao["horas_indisponiveis"] > 0
).sum()

print(f"Total de dias: {total_dias:,}")
print(f"Dias completos: {dias_completos:,}")
print(f"Dias incompletos: {dias_incompletos:,}")
print(f"Percentual completo: {dias_completos / total_dias * 100:.2f}%")

Total de dias: 5,844
Dias completos: 5,655
Dias incompletos: 189
Percentual completo: 96.77%


In [25]:
distribuicao_horas_faltantes = (
    qualidade_diaria_precipitacao.loc[
        qualidade_diaria_precipitacao["horas_indisponiveis"] > 0,
        "horas_indisponiveis"
    ]
    .value_counts()
    .sort_index()
)

distribuicao_horas_faltantes

horas_indisponiveis
1     103
2      17
3      11
4       3
5       2
6       1
7       2
9       3
10      1
11      3
12      1
13      2
14      2
15      4
16      1
17      4
18      1
21      1
24     27
Name: count, dtype: int64

In [26]:
def classificar_falha(horas):
    if horas == 0:
        return "Completo"
    elif horas <= 2:
        return "1-2 horas"
    elif horas <= 6:
        return "3-6 horas"
    elif horas <= 12:
        return "7-12 horas"
    elif horas <= 23:
        return "13-23 horas"
    else:
        return "24 horas"

qualidade_diaria_precipitacao["faixa_falha"] = (
    qualidade_diaria_precipitacao["horas_indisponiveis"]
    .apply(classificar_falha)
)

qualidade_diaria_precipitacao["faixa_falha"].value_counts()

faixa_falha
Completo       5655
1-2 horas       120
24 horas         27
3-6 horas        17
13-23 horas      15
7-12 horas       10
Name: count, dtype: int64

### Critério de completude da precipitação diária

A precipitação diária foi calculada apenas para dias com 24 observações horárias válidas. Essa regra foi adotada porque a ausência de uma ou mais horas pode subestimar o acumulado diário, especialmente quando a chuva ocorre de forma concentrada em períodos curtos.

Com esse critério, foram obtidos 5.655 dias completos e 189 dias incompletos, correspondendo a 96,77% de disponibilidade da variável-alvo.

In [27]:
df["precipitacao_horaria"] = (
    df[col_precipitacao]
    .replace(-9999, np.nan)
)

In [28]:
print(
    "Valores -9999 restantes:",
    (df["precipitacao_horaria"] == -9999).sum()
)

print(
    "Valores ausentes:",
    df["precipitacao_horaria"].isna().sum()
)

Valores -9999 restantes: 0
Valores ausentes: 1179


In [29]:
precipitacao_diaria = (
    df.groupby("DATA (YYYY-MM-DD)")
    .agg(
        horas_validas=(
            "precipitacao_horaria",
            "count"
        ),
        precipitacao_diaria_mm=(
            "precipitacao_horaria",
            lambda x: x.sum(min_count=24)
        )
    )
    .reset_index()
)

In [30]:
precipitacao_diaria["dia_completo"] = (
    precipitacao_diaria["horas_validas"] == 24
)

precipitacao_diaria["dia_completo"].value_counts()

dia_completo
True     5655
False     189
Name: count, dtype: int64

## 7. Avaliação das variáveis auxiliares

Após o tratamento da precipitação, são avaliadas as principais variáveis meteorológicas auxiliares: temperatura, umidade relativa, pressão atmosférica e velocidade do vento. A radiação global é analisada separadamente devido ao seu padrão específico de disponibilidade ao longo do dia.


In [31]:
variaveis_auxiliares = {
    "temperatura_horaria": "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "umidade_horaria": "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "pressao_horaria": "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "vento_horaria": "VENTO, VELOCIDADE HORARIA (m/s)",
    "radiacao_horaria": "RADIACAO GLOBAL (KJ/m²)"
}

for nova_coluna, coluna_original in variaveis_auxiliares.items():
    df[nova_coluna] = df[coluna_original].replace(-9999, np.nan)

In [32]:
resumo_auxiliares = pd.DataFrame({
    "nulos": df[list(variaveis_auxiliares.keys())].isna().sum(),
    "percentual_nulos": (
        df[list(variaveis_auxiliares.keys())].isna().mean() * 100
    ).round(2)
})

resumo_auxiliares

,nulos,percentual_nulos
temperatura_horaria,893,0.64
umidade_horaria,742,0.53
pressao_horaria,740,0.53
vento_horaria,743,0.53
radiacao_horaria,65956,47.03


In [33]:
df["hora"] = df["datetime"].dt.hour

radiacao_por_hora = (
    df.groupby("hora")["radiacao_horaria"]
    .agg(
        total_registros="size",
        registros_validos="count"
    )
)

radiacao_por_hora["indisponiveis"] = (
    radiacao_por_hora["total_registros"]
    - radiacao_por_hora["registros_validos"]
)

radiacao_por_hora["percentual_indisponivel"] = (
    radiacao_por_hora["indisponiveis"]
    / radiacao_por_hora["total_registros"]
    * 100
).round(2)

radiacao_por_hora

,total_registros,registros_validos,indisponiveis,percentual_indisponivel
hora,,,,
0,5844,125,5719,97.86
1,5844,98,5746,98.32
2,5844,95,5749,98.37
3,5844,81,5763,98.61
4,5844,79,5765,98.65
5,5844,67,5777,98.85
6,5844,86,5758,98.53
7,5844,82,5762,98.60
8,5844,145,5699,97.52


In [34]:
radiacao_original = "RADIACAO GLOBAL (KJ/m²)"

radiacao_tipo_falha_hora = pd.DataFrame({
    "nulos": (
        df[radiacao_original]
        .isna()
        .groupby(df["hora"])
        .sum()
    ),
    "codigo_-9999": (
        (df[radiacao_original] == -9999)
        .groupby(df["hora"])
        .sum()
    )
})

radiacao_tipo_falha_hora["total"] = (
    radiacao_tipo_falha_hora["nulos"]
    + radiacao_tipo_falha_hora["codigo_-9999"]
)

radiacao_tipo_falha_hora

,nulos,codigo_-9999,total
hora,,,
0,2500,3219,5719
1,2512,3234,5746
2,2517,3232,5749
3,2519,3244,5763
4,2529,3236,5765
5,2531,3246,5777
6,2519,3239,5758
7,2524,3238,5762
8,2513,3186,5699


In [35]:
radiacao_valida_por_hora = (
    df.groupby("hora")["radiacao_horaria"]
    .agg(
        quantidade="count",
        media="mean",
        mediana="median",
        minimo="min",
        maximo="max"
    )
    .round(2)
)

radiacao_valida_por_hora

,quantidade,media,mediana,minimo,maximo
hora,,,,,
0,125,1.11,0.70,0.0,6.7
1,98,0.85,0.60,0.0,4.0
2,95,0.74,0.50,0.0,6.1
3,81,0.74,0.50,0.0,4.4
4,79,0.73,0.40,0.0,4.7
5,67,0.70,0.40,0.0,3.2
6,86,0.79,0.50,0.0,3.3
7,82,0.56,0.30,0.0,2.5
8,145,1.41,0.60,0.0,13.4


### Avaliação da radiação global

A radiação global apresentou 47,03% de registros indisponíveis na série horária. Entretanto, a análise por horário evidenciou forte concentração das ausências durante o período noturno, enquanto as horas diurnas apresentaram elevada disponibilidade e valores coerentes com o ciclo diário de radiação solar. Devido a esse padrão estrutural de ausência e visando evitar imputações baseadas em pressupostos não observados diretamente nos dados, a variável não será considerada inicialmente entre os preditores meteorológicos principais, permanecendo disponível para análises experimentais posteriores.


## 8. Validação temporal e preparação da base diária

Antes da consolidação final, é verificada a integridade da sequência temporal e são preparadas as estatísticas diárias das variáveis auxiliares. Também são aplicados os critérios de completude e realizadas verificações básicas de valores inválidos.


In [36]:
duplicados_datetime = df["datetime"].duplicated().sum()

print("Timestamps duplicados:", duplicados_datetime)

Timestamps duplicados: 0


In [37]:
datas_ordenadas = df["datetime"].sort_values()

diferencas = datas_ordenadas.diff()

diferencas.value_counts().head(10)

datetime
0 days 01:00:00    140255
Name: count, dtype: int64

### Validação da continuidade temporal

A verificação da continuidade temporal não identificou timestamps duplicados nem lacunas na sequência horária. As 140.255 diferenças entre registros consecutivos foram de exatamente uma hora, indicando que a série temporal cobre integralmente o período analisado. Dessa forma, as indisponibilidades observadas referem-se aos valores das variáveis meteorológicas, e não à ausência de registros horários na estrutura da base.


In [38]:
variaveis_diarias = {
    "temperatura_horaria": "temperatura_media",
    "umidade_horaria": "umidade_media",
    "pressao_horaria": "pressao_media",
    "vento_horaria": "vento_medio"
}

agregacoes = {}

for coluna_horaria, coluna_diaria in variaveis_diarias.items():
    agregacoes[coluna_diaria] = (coluna_horaria, "mean")
    agregacoes[f"{coluna_diaria}_horas_validas"] = (coluna_horaria, "count")

auxiliares_diarias = (
    df.groupby("DATA (YYYY-MM-DD)")
    .agg(**agregacoes)
    .reset_index()
)

auxiliares_diarias.head()

,DATA (YYYY-MM-DD),temperatura_media,temperatura_media_horas_validas,umidade_media,umidade_media_horas_validas,pressao_media,pressao_media_horas_validas,vento_medio,vento_medio_horas_validas
0,2010-01-01,21.333333,24,86.916667,24,923.025000,24,2.329167,24
1,2010-01-02,22.166667,24,83.833333,24,925.041667,24,1.079167,24
2,2010-01-03,24.229167,24,78.625000,24,926.712500,24,1.433333,24
3,2010-01-04,24.241667,24,74.500000,24,926.695833,24,2.004167,24
4,2010-01-05,24.845833,24,72.666667,24,924.004167,24,1.762500,24


In [39]:
colunas_horas_validas = [
    "temperatura_media_horas_validas",
    "umidade_media_horas_validas",
    "pressao_media_horas_validas",
    "vento_medio_horas_validas"
]

auxiliares_diarias[colunas_horas_validas].describe()

,temperatura_media_horas_validas,umidade_media_horas_validas,pressao_media_horas_validas,vento_medio_horas_validas
count,5844.000000,5844.000000,5844.000000,5844.000000
mean,23.847194,23.873032,23.873374,23.872861
std,1.551144,1.345741,1.345647,1.350993
min,0.000000,0.000000,0.000000,0.000000
25%,24.000000,24.000000,24.000000,24.000000
50%,24.000000,24.000000,24.000000,24.000000
75%,24.000000,24.000000,24.000000,24.000000
max,24.000000,24.000000,24.000000,24.000000


In [40]:
distribuicao_horas_validas = pd.DataFrame({
    coluna: auxiliares_diarias[coluna]
        .value_counts()
        .sort_index()
    for coluna in colunas_horas_validas
}).fillna(0).astype(int)

distribuicao_horas_validas

,temperatura_media_horas_validas,umidade_media_horas_validas,pressao_media_horas_validas,vento_medio_horas_validas
0,17,11,11,11
6,1,1,1,2
7,4,4,4,3
9,4,4,4,4
10,2,2,2,3
11,2,2,2,1
12,1,1,1,2
13,3,3,3,2
15,3,3,3,3
16,1,0,0,0


In [41]:
auxiliares_diarias_tratadas = auxiliares_diarias.copy()

In [42]:
regras_auxiliares = {
    "temperatura_media": "temperatura_media_horas_validas",
    "umidade_media": "umidade_media_horas_validas",
    "pressao_media": "pressao_media_horas_validas",
    "vento_medio": "vento_medio_horas_validas"
}

for variavel, coluna_horas in regras_auxiliares.items():
    auxiliares_diarias_tratadas.loc[
        auxiliares_diarias_tratadas[coluna_horas] < 20,
        variavel
    ] = np.nan

In [43]:
resumo_auxiliares_diarias = pd.DataFrame({
    "dias_validos": auxiliares_diarias_tratadas[
        list(regras_auxiliares.keys())
    ].notna().sum(),

    "dias_indisponiveis": auxiliares_diarias_tratadas[
        list(regras_auxiliares.keys())
    ].isna().sum()
})

resumo_auxiliares_diarias["percentual_disponivel"] = (
    resumo_auxiliares_diarias["dias_validos"]
    / len(auxiliares_diarias_tratadas)
    * 100
).round(2)

resumo_auxiliares_diarias

,dias_validos,dias_indisponiveis,percentual_disponivel
temperatura_media,5802,42,99.28
umidade_media,5809,35,99.40
pressao_media,5809,35,99.40
vento_medio,5809,35,99.40


### Critério de completude das variáveis auxiliares

Para temperatura do ar, umidade relativa, pressão atmosférica e velocidade do vento, foram calculadas médias diárias a partir dos registros horários válidos. Diferentemente da precipitação, cuja agregação diária corresponde à soma das observações e exige completude integral, as variáveis auxiliares foram consideradas representativas quando apresentaram pelo menos 20 das 24 observações horárias válidas no dia, correspondendo a uma completude mínima de 83,3%.

Nos dias abaixo desse limite, o valor diário foi mantido como ausente. Não foram realizadas imputações nesta etapa, de modo a evitar a introdução antecipada de valores artificiais e eventual vazamento de informação entre os conjuntos de treinamento e avaliação.

In [44]:
variaveis_auditoria = [
    "precipitacao_horaria",
    "temperatura_horaria",
    "umidade_horaria",
    "pressao_horaria",
    "vento_horaria"
]

df[variaveis_auditoria].describe().T

,count,mean,std,min,25%,50%,75%,max
precipitacao_horaria,139077.0,0.179650,1.438464,0.0,0.0,0.0,0.0,82.0
temperatura_horaria,139363.0,20.538624,4.592903,4.1,17.3,20.3,23.4,37.3
umidade_horaria,139514.0,69.306464,17.835430,12.0,57.0,74.0,83.0,100.0
pressao_horaria,139516.0,927.155068,3.640624,913.1,924.6,927.0,929.6,940.3
vento_horaria,139513.0,1.845925,1.186329,0.0,1.0,1.8,2.6,10.9


In [45]:
print(
    "Precipitação negativa:",
    (df["precipitacao_horaria"] < 0).sum()
)

print(
    "Umidade fora de 0–100%:",
    (
        (df["umidade_horaria"] < 0)
        | (df["umidade_horaria"] > 100)
    ).sum()
)

print(
    "Vento negativo:",
    (df["vento_horaria"] < 0).sum()
)

Precipitação negativa: 0
Umidade fora de 0–100%: 0
Vento negativo: 0


### Verificação de valores inválidos

Não foram identificados valores negativos de precipitação ou velocidade do vento, nem registros de umidade relativa fora do intervalo de 0% a 100%.

Os valores mínimos e máximos observados para temperatura e pressão atmosférica também não apresentaram inconsistências evidentes. Dessa forma, não foram aplicados cortes adicionais baseados em limites arbitrários.

In [46]:
df_diario = precipitacao_diaria.merge(
    auxiliares_diarias_tratadas,
    on="DATA (YYYY-MM-DD)",
    how="left"
)

df_diario.head()

,DATA (YYYY-MM-DD),horas_validas,precipitacao_diaria_mm,dia_completo,temperatura_media,temperatura_media_horas_validas,umidade_media,umidade_media_horas_validas,pressao_media,pressao_media_horas_validas,vento_medio,vento_medio_horas_validas
0,2010-01-01,24,6.8,True,21.333333,24,86.916667,24,923.025000,24,2.329167,24
1,2010-01-02,24,36.2,True,22.166667,24,83.833333,24,925.041667,24,1.079167,24
2,2010-01-03,24,0.8,True,24.229167,24,78.625000,24,926.712500,24,1.433333,24
3,2010-01-04,24,13.4,True,24.241667,24,74.500000,24,926.695833,24,2.004167,24
4,2010-01-05,24,20.8,True,24.845833,24,72.666667,24,924.004167,24,1.762500,24


In [47]:
print("Linhas:", len(df_diario))
print("Data mínima:", df_diario["DATA (YYYY-MM-DD)"].min())
print("Data máxima:", df_diario["DATA (YYYY-MM-DD)"].max())

Linhas: 5844
Data mínima: 2010-01-01 00:00:00
Data máxima: 2025-12-31 00:00:00


In [48]:
colunas_modelagem = [
    "precipitacao_diaria_mm",
    "temperatura_media",
    "umidade_media",
    "pressao_media",
    "vento_medio"
]

resumo_diario = pd.DataFrame({
    "nulos": df_diario[colunas_modelagem].isna().sum(),
    "percentual_nulos": (
        df_diario[colunas_modelagem].isna().mean() * 100
    ).round(2)
})

resumo_diario

,nulos,percentual_nulos
precipitacao_diaria_mm,189,3.23
temperatura_media,42,0.72
umidade_media,35,0.60
pressao_media,35,0.60
vento_medio,35,0.60


## 9. Validação e caracterização da base diária

Após as etapas de padronização, avaliação da qualidade, tratamento dos códigos de ausência e agregação temporal, foi obtida uma base diária consolidada para o período de 2010 a 2025.

As tabelas a seguir sintetizam as principais características da base resultante e a disponibilidade das variáveis que serão utilizadas nas etapas posteriores de análise exploratória e modelagem.


In [49]:
resumo_base_final = pd.DataFrame({
    "Característica": [
        "Estação",
        "Período",
        "Frequência original",
        "Registros horários",
        "Frequência final",
        "Dias da série",
        "Variável-alvo",
        "Variáveis auxiliares principais"
    ],
    "Resultado": [
        "A701 - Mirante de Santana",
        "2010-2025",
        "Horária",
        f"{len(df):,}".replace(",", "."),
        "Diária",
        f"{len(df_diario):,}".replace(",", "."),
        "Precipitação diária acumulada",
        "Temperatura, umidade, pressão e vento"
    ]
})

resumo_base_final

,Característica,Resultado
0,Estação,A701 - Mirante de Santana
1,Período,2010-2025
2,Frequência original,Horária
3,Registros horários,140.256
4,Frequência final,Diária
5,Dias da série,5.844
6,Variável-alvo,Precipitação diária acumulada
7,Variáveis auxiliares principais,"Temperatura, umidade, pressão e vento"


In [50]:
resumo_qualidade_final = pd.DataFrame({
    "Variável": [
        "Precipitação diária",
        "Temperatura média",
        "Umidade média",
        "Pressão média",
        "Vento médio"
    ],
    "Dias válidos": [
        df_diario["precipitacao_diaria_mm"].notna().sum(),
        df_diario["temperatura_media"].notna().sum(),
        df_diario["umidade_media"].notna().sum(),
        df_diario["pressao_media"].notna().sum(),
        df_diario["vento_medio"].notna().sum()
    ],
    "Dias indisponíveis": [
        df_diario["precipitacao_diaria_mm"].isna().sum(),
        df_diario["temperatura_media"].isna().sum(),
        df_diario["umidade_media"].isna().sum(),
        df_diario["pressao_media"].isna().sum(),
        df_diario["vento_medio"].isna().sum()
    ]
})

resumo_qualidade_final["Disponibilidade (%)"] = (
    resumo_qualidade_final["Dias válidos"]
    / len(df_diario)
    * 100
).round(2)

resumo_qualidade_final

,Variável,Dias válidos,Dias indisponíveis,Disponibilidade (%)
0,Precipitação diária,5655,189,96.77
1,Temperatura média,5802,42,99.28
2,Umidade média,5809,35,99.40
3,Pressão média,5809,35,99.40
4,Vento médio,5809,35,99.40


## 10. Exportação para a camada Gold

Após a validação da base diária, o conjunto consolidado é armazenado em formato Parquet na camada Gold. Essa base será utilizada como entrada para as etapas seguintes de análise exploratória, engenharia de atributos e modelagem.


In [51]:
from pathlib import Path

pasta_gold = Path("../data/gold")
pasta_gold.mkdir(parents=True, exist_ok=True)

arquivo_gold = pasta_gold / "a701_2010_2025_diario.parquet"

df_diario.to_parquet(
    arquivo_gold,
    index=False
)

print(f"Arquivo salvo em: {arquivo_gold}")

Arquivo salvo em: ..\data\gold\a701_2010_2025_diario.parquet


## 11. Síntese da etapa

A etapa de qualidade e pré-processamento resultou em uma base diária consolidada contendo 5.844 dias entre 2010 e 2025.

Para a precipitação, foram considerados válidos apenas os dias com 24 observações horárias disponíveis, resultando em 5.655 dias completos e 189 dias com precipitação diária indisponível.

Para temperatura, umidade relativa, pressão atmosférica e velocidade do vento, foram calculadas médias diárias apenas quando havia pelo menos 20 das 24 observações horárias válidas. Após a aplicação desse critério, as variáveis auxiliares apresentaram disponibilidade superior a 99%.

Não foram identificados timestamps duplicados ou lacunas na estrutura temporal da série. Também não foram encontrados valores negativos de precipitação ou velocidade do vento, nem valores de umidade fora do intervalo entre 0% e 100%.

A radiação global apresentou padrão de ausência fortemente relacionado ao horário e, por esse motivo, não foi incluída inicialmente entre os preditores meteorológicos principais, permanecendo disponível para análises experimentais posteriores.

A base diária resultante foi armazenada na camada Gold e será utilizada nas etapas seguintes de análise exploratória, engenharia de atributos e modelagem preditiva.
